In [ ]:
# ═══════════════════════════════════════════════════
# 1  SETUP PROJET (Kaggle Dataset → /kaggle/working)
# ═══════════════════════════════════════════════════
# Sur Kaggle : ajoute ton dataset via l'onglet "Data"
# Chemin attendu : /kaggle/input/<nom-dataset>/Vision_Project.zip

import os, zipfile, shutil, sys

# Trouver le zip automatiquement dans /kaggle/input/
INPUT_BASE = '/kaggle/input'
ZIP_PATH   = None

for root, dirs, files in os.walk(INPUT_BASE):
    for f in files:
        if f == 'Vision_Project.zip':
            ZIP_PATH = os.path.join(root, f)
            break

if ZIP_PATH is None:
    raise FileNotFoundError(
        "Vision_Project.zip introuvable dans /kaggle/input/\n"
        "Ajoute ton dataset via l'onglet Data du kernel Kaggle."
    )

DEST_DIR = '/kaggle/working/Vision_Project'

if os.path.isdir(DEST_DIR):
    shutil.rmtree(DEST_DIR)
os.makedirs(DEST_DIR, exist_ok=True)

with zipfile.ZipFile(ZIP_PATH, 'r') as z:
    z.extractall(DEST_DIR)

print('Extraction OK ->', DEST_DIR)
print('Contenu :')
for item in os.listdir(DEST_DIR):
    print(' ', item)


In [ ]:
# ═══════════════════════════════════════════════════
# 2  DEPENDANCES
# ═══════════════════════════════════════════════════
# Kaggle fournit torch + torchvision préinstallés avec CUDA — ne pas réinstaller.
# On installe uniquement ce qui manque.

import subprocess, sys

def pip(pkg):
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', pkg], check=True)

pip('Pillow==10.2.0')        # facenet-pytorch exige Pillow < 10.3
pip('facenet-pytorch==2.5.3')
pip('ultralytics')
pip('flask')

import PIL, torch, torchvision, facenet_pytorch
print('Pillow      :', PIL.__version__)
print('torch       :', torch.__version__)
print('torchvision :', torchvision.__version__)
print('CUDA dispo  :', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU         :', torch.cuda.get_device_name(0))
print('Dependances OK')


In [ ]:
# ═══════════════════════════════════════════════════
# 3  PYTHONPATH + __init__.py
# ═══════════════════════════════════════════════════
import sys, pathlib

PROJECT_ROOT = '/kaggle/working/Vision_Project/Vision_Project'
if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

for pkg in ['securai_store', 'securai_store/modules']:
    init_file = pathlib.Path(PROJECT_ROOT) / pkg / '__init__.py'
    if not init_file.exists():
        init_file.touch()
        print('Cree :', init_file)
    else:
        print('Deja present :', init_file)

print('PYTHONPATH ->', PROJECT_ROOT)


In [ ]:
# ═══════════════════════════════════════════════════
# 4  TEST D'IMPORT
# ═══════════════════════════════════════════════════
try:
    from securai_store.modules.face_recognizer import FaceRecognizer
    from securai_store.modules.patch_attacker  import PatchAttacker
    print('FaceRecognizer OK')
    print('PatchAttacker  OK')
except Exception as e:
    print('Import echoue :', e)
    import os
    print('Contenu de', '/kaggle/working/Vision_Project:')
    for item in os.listdir('/kaggle/working/Vision_Project'):
        print(' ', item)


In [ ]:
# ═══════════════════════════════════════════════════
# 5  INSTALLER CLOUDFLARED
# ═══════════════════════════════════════════════════
# cloudflared n'est pas preinstalle sur Kaggle — on le recupere depuis GitHub.

import subprocess, os

CLOUDFLARED_BIN = '/usr/local/bin/cloudflared'

if not os.path.exists(CLOUDFLARED_BIN):
    print('Telechargement de cloudflared...')
    subprocess.run([
        'wget', '-q',
        'https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64',
        '-O', CLOUDFLARED_BIN
    ], check=True)
    os.chmod(CLOUDFLARED_BIN, 0o755)
    print('cloudflared installe ->', CLOUDFLARED_BIN)
else:
    print('cloudflared deja present')

result = subprocess.run([CLOUDFLARED_BIN, '--version'], capture_output=True, text=True)
print(result.stdout.strip())


In [ ]:
# ═══════════════════════════════════════════════════
# 6  FLASK API + CLOUDFLARE TUNNEL
# ═══════════════════════════════════════════════════
# Le token Cloudflare est lu depuis les Secrets Kaggle (plus securise).
# Dans Kaggle : Settings -> Add-ons -> Secrets -> ajoute CF_TOKEN

import sys
sys.path.insert(0, '/kaggle/working/Vision_Project/Vision_Project')

import os, subprocess, threading, time
import torch, cv2, base64, numpy as np
from flask import Flask, request, jsonify
from securai_store.modules.face_recognizer import FaceRecognizer

# --- Config ---
PUBLIC_HOSTNAME = 'vision.api.near-u-api.org'
FLASK_PORT      = 9999

# Lecture du token depuis les Secrets Kaggle (recommande)
# Si pas de secret configure, fallback sur variable d'env ou valeur directe
try:
    from kaggle_secrets import UserSecretsClient
    CF_TOKEN = UserSecretsClient().get_secret('CF_TOKEN')
    print('[INFO] Token lu depuis Kaggle Secrets')
except Exception:
    # Fallback : colle ton token ici uniquement en local/test
    CF_TOKEN = os.environ.get('CF_TOKEN', 'COLLE_TON_TOKEN_ICI')
    print('[WARN] Kaggle Secrets non disponible — token depuis env ou fallback')

# --- Liberer le port si occupe ---
try:
    pids = subprocess.check_output(
        ['lsof', '-t', f'-i:{FLASK_PORT}'], text=True
    ).strip().split()
    for pid in pids:
        subprocess.run(['kill', '-9', pid])
    print(f'[INFO] Port {FLASK_PORT} libere')
except Exception:
    print(f'[INFO] Port {FLASK_PORT} libre')

# --- Device ---
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'[INFO] Device: {device}')
if device == 'cuda':
    print(f'[INFO] GPU: {torch.cuda.get_device_name(0)}')

# --- Modele ---
face_rec = FaceRecognizer()
face_rec.model = face_rec.model.to(device)
print('[INFO] FaceRecognizer charge et pret')

# --- Flask ---
app = Flask(__name__)

def _decode_image(b64_string: str) -> np.ndarray:
    img_bytes = base64.b64decode(b64_string.split(',')[-1])
    np_arr    = np.frombuffer(img_bytes, np.uint8)
    bgr_img   = cv2.imdecode(np_arr, cv2.IMREAD_COLOR)
    return cv2.cvtColor(bgr_img, cv2.COLOR_BGR2RGB)

@app.route('/infer', methods=['POST'])
def infer():
    try:
        payload = request.get_json()
        if not payload or 'image' not in payload:
            return jsonify({'error': "Missing 'image' field"}), 400
        img_rgb = _decode_image(payload['image'])
        with torch.no_grad():
            name, confidence = face_rec.predict(img_rgb)
        if name in ['Unknown', 'Inconnu']:
            access, name = 'DENIED', 'Inconnu'
        else:
            access = 'GRANTED'
        return jsonify({'name': name, 'confidence': round(float(confidence), 4), 'access': access})
    except Exception as e:
        return jsonify({'error': str(e)}), 400

@app.route('/health', methods=['GET'])
def health():
    return jsonify({'status': 'ok', 'device': device,
                    'gpu': torch.cuda.get_device_name(0) if device == 'cuda' else 'none'})

flask_thread = threading.Thread(
    target=lambda: app.run(host='0.0.0.0', port=FLASK_PORT, debug=False, use_reloader=False),
    daemon=True
)
flask_thread.start()
time.sleep(2)
print(f'[INFO] Flask demarre sur le port {FLASK_PORT}')

# --- Cloudflare Tunnel ---
# Syntaxe correcte avec --token : on passe uniquement run --token <TOKEN>
# --url n'est pas compatible avec run --token
_cf_ready = threading.Event()

def _run_cloudflared():
    cmd = [
        '/usr/local/bin/cloudflared', 'tunnel', 'run',
        '--token', CF_TOKEN
    ]
    proc = subprocess.Popen(
        cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True
    )
    globals()['_cf_proc'] = proc
    for line in iter(proc.stdout.readline, ''):
        print('[CF]', line.rstrip())
        if any(kw in line for kw in ['Registered tunnel', 'Connection established', 'connnected']):
            print(f'\nTunnel READY -> https://{PUBLIC_HOSTNAME}')
            _cf_ready.set()
            break

threading.Thread(target=_run_cloudflared, daemon=True).start()

# Attendre max 15s que le tunnel soit up
_cf_ready.wait(timeout=15)
time.sleep(2)

proc = globals().get('_cf_proc')
if proc and proc.poll() is None:
    print(f'[INFO] Tunnel actif (PID={proc.pid})')
    print(f'[INFO] POST -> https://{PUBLIC_HOSTNAME}/infer')
    print(f'[INFO] GET  -> https://{PUBLIC_HOSTNAME}/health')
else:
    print('[WARN] Tunnel non demarre ou crash — relance la cellule et verifie CF_TOKEN')


In [ ]:
# ═══════════════════════════════════════════════════
# 7  MONITORING DES REQUETES
# ═══════════════════════════════════════════════════
import time as _time

_original_infer = app.view_functions['infer']
_count = 0

def _monitored_infer():
    global _count
    _count += 1
    print(f'[{_count}] /infer recu a {_time.strftime("%H:%M:%S")}')
    return _original_infer()

app.view_functions['infer'] = _monitored_infer
print('Monitoring actif — chaque requete /infer sera loggee ici')
